# FinanceRAG Analyst — 01 Corpus Ingestion
#This notebook builds a deterministic and auditable corpus of SEC EDGAR filings for FinanceRAG-Analyst.

## Objectives

1. Define the corpus using exact SEC accession numbers.
2. Download or load every filing reproducibly.
3. preserve filing-level provenance and SEC source URLs;
4. assign deterministic document identifiers;
5. calculate content hashes for corpus versioning;
6. validate document completeness and uniqueness;
7. produce a lightweight corpus manifest for downstream processing.

## Reproducibility rule

The corpus is not defined as the filings that are “latest” at execution time.

Every accepted filing must be identified by a fixed accession number so that future executions reconstruct the same corpus.

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

NOTEBOOK_NAME = "01_corpus_ingestion.ipynb"
PROJECT_NAME = "FinanceRAG-Analyst"
RANDOM_SEED = 42

TARGET_TICKERS = ["NVDA", "MSFT", "AAPL", "AMZN", "GOOGL"]
ALLOWED_FORMS = {"10-K", "10-Q"}

SEC_REQUEST_TIMEOUT_SECONDS = 30
SEC_REQUEST_DELAY_SECONDS = 0.15
MIN_DOCUMENT_CHARACTERS = 20_000

print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"UTC run time: {datetime.now(timezone.utc).isoformat()}")

Notebook: 01_corpus_ingestion.ipynb
Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
UTC run time: 2026-08-05T06:10:17.179075+00:00


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Locate existing FinanceRAG data

In [ ]:
drive_root = Path("/content/drive/MyDrive")

project_dirs = sorted(
    path
    for path in drive_root.rglob("*")
    if path.is_dir()
    and (
        "financerag" in path.name.lower()
        or "finance_rag" in path.name.lower()
        or "finrag" in path.name.lower()
    )
)

print(f"Possible project directories found: {len(project_dirs)}")

for path in project_dirs[:50]:
    print(path)

Possible project directories found: 0


In [ ]:
candidate_files = sorted(
    path
    for path in drive_root.rglob("*")
    if path.is_file()
    and (
        "sec_filings" in path.name.lower()
        or "filings" in path.name.lower()
        or "chunks" in path.name.lower()
        or "manifest" in path.name.lower()
    )
)

print(f"Possible corpus files found: {len(candidate_files)}")

for path in candidate_files[:100]:
    print(f"{path} | {path.stat().st_size:,} bytes")

Possible corpus files found: 7
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filing_chunks.csv | 8,897,636 bytes
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filing_chunks.parquet | 3,415,080 bytes
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filings_raw_final.csv | 6,769,519 bytes
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filings_raw_final.parquet | 3,066,858 bytes
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filings_sections.csv | 6,634,152 bytes
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filings_sections_v2.csv | 6,610,929 bytes
/content/drive/MyDrive/finance-rag-analyst/data/raw/sec_filings_raw.csv | 6,769,112 bytes


In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/finance-rag-analyst")
LEGACY_RAW_CSV = PROJECT_ROOT / "data/raw/sec_filings_raw.csv"

assert PROJECT_ROOT.exists(), f"Project folder not found: {PROJECT_ROOT}"
assert LEGACY_RAW_CSV.exists(), f"Legacy corpus not found: {LEGACY_RAW_CSV}"

legacy_raw = pd.read_csv(LEGACY_RAW_CSV)

print(f"Rows: {len(legacy_raw):,}")
print(f"Columns: {len(legacy_raw.columns)}")
print("\nColumn names:")
for column in legacy_raw.columns:
    print(f"- {column}")

print("\nTicker/form counts:")
display(
    legacy_raw.groupby(["ticker", "form_type"], dropna=False)
    .size()
    .rename("document_count")
    .reset_index()
)

Rows: 30
Columns: 10

Column names:
- ticker
- company
- cik
- form_type
- filing_date
- report_date
- accession_number
- primary_document
- source_url
- text

Ticker/form counts:


,ticker,form_type,document_count
0,AAPL,10-K,3
1,AAPL,10-Q,3
2,AMZN,10-K,3
3,AMZN,10-Q,3
4,GOOGL,10-K,3
5,GOOGL,10-Q,3
6,MSFT,10-K,3
7,MSFT,10-Q,3
8,NVDA,10-K,3
9,NVDA,10-Q,3


In [ ]:
metadata_columns = [
    "ticker",
    "company",
    "cik",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "primary_document",
    "source_url",
]

display(legacy_raw[metadata_columns].head(10))

,ticker,company,cik,form_type,filing_date,report_date,accession_number,primary_document,source_url
0,NVDA,NVIDIA CORP,1045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,https://www.sec.gov/Archives/edgar/data/104581...
1,NVDA,NVIDIA CORP,1045810,10-K,2025-02-26,2025-01-26,0001045810-25-000023,nvda-20250126.htm,https://www.sec.gov/Archives/edgar/data/104581...
2,NVDA,NVIDIA CORP,1045810,10-K,2024-02-21,2024-01-28,0001045810-24-000029,nvda-20240128.htm,https://www.sec.gov/Archives/edgar/data/104581...
3,NVDA,NVIDIA CORP,1045810,10-Q,2026-05-20,2026-04-26,0001045810-26-000052,nvda-20260426.htm,https://www.sec.gov/Archives/edgar/data/104581...
4,NVDA,NVIDIA CORP,1045810,10-Q,2025-11-19,2025-10-26,0001045810-25-000230,nvda-20251026.htm,https://www.sec.gov/Archives/edgar/data/104581...
5,NVDA,NVIDIA CORP,1045810,10-Q,2025-08-27,2025-07-27,0001045810-25-000209,nvda-20250727.htm,https://www.sec.gov/Archives/edgar/data/104581...
6,AAPL,Apple Inc.,320193,10-K,2025-10-31,2025-09-27,0000320193-25-000079,aapl-20250927.htm,https://www.sec.gov/Archives/edgar/data/320193...
7,AAPL,Apple Inc.,320193,10-K,2024-11-01,2024-09-28,0000320193-24-000123,aapl-20240928.htm,https://www.sec.gov/Archives/edgar/data/320193...
8,AAPL,Apple Inc.,320193,10-K,2023-11-03,2023-09-30,0000320193-23-000106,aapl-20230930.htm,https://www.sec.gov/Archives/edgar/data/320193...
9,AAPL,Apple Inc.,320193,10-Q,2026-05-01,2026-03-28,0000320193-26-000013,aapl-20260328.htm,https://www.sec.gov/Archives/edgar/data/320193...


In [ ]:
required_columns = [
    "ticker",
    "company",
    "cik",
    "form_type",
    "filing_date",
    "accession_number",
    "primary_document",
    "source_url",
    "text",
]

missing_columns = [
    column
    for column in required_columns
    if column not in legacy_raw.columns
]

assert not missing_columns, f"Missing required columns: {missing_columns}"

print("Missing values by required column:")
display(
    legacy_raw[required_columns]
    .isna()
    .sum()
    .rename("missing_count")
)

print(
    "Duplicate accession numbers:",
    legacy_raw["accession_number"].duplicated().sum(),
)
print(
    "Duplicate source URLs:",
    legacy_raw["source_url"].duplicated().sum(),
)

Missing values by required column:


,missing_count
ticker,0
company,0
cik,0
form_type,0
filing_date,0
accession_number,0
primary_document,0
source_url,0
text,0


Duplicate accession numbers: 0
Duplicate source URLs: 0


normailsation and validation of metadata

In [ ]:
frozen_corpus = legacy_raw.copy()

# Normalize identifiers and categorical metadata.
frozen_corpus["ticker"] = (
    frozen_corpus["ticker"].astype("string").str.strip().str.upper()
)

frozen_corpus["form_type"] = (
    frozen_corpus["form_type"].astype("string").str.strip().str.upper()
)

frozen_corpus["company"] = (
    frozen_corpus["company"].astype("string").str.strip()
)

frozen_corpus["cik"] = (
    frozen_corpus["cik"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
    .str.zfill(10)
)

frozen_corpus["accession_number"] = (
    frozen_corpus["accession_number"]
    .astype("string")
    .str.strip()
)

frozen_corpus["primary_document"] = (
    frozen_corpus["primary_document"].astype("string").str.strip()
)

frozen_corpus["source_url"] = (
    frozen_corpus["source_url"].astype("string").str.strip()
)

# Parse dates without silently accepting invalid values.
for date_column in ["filing_date", "report_date"]:
    frozen_corpus[date_column] = pd.to_datetime(
        frozen_corpus[date_column],
        errors="raise",
    ).dt.strftime("%Y-%m-%d")

assert set(frozen_corpus["ticker"]) == set(TARGET_TICKERS)
assert set(frozen_corpus["form_type"]).issubset(ALLOWED_FORMS)
assert frozen_corpus["accession_number"].is_unique
assert frozen_corpus["source_url"].is_unique

print("Normalized corpus metadata successfully.")
display(
    frozen_corpus[
        [
            "ticker",
            "company",
            "cik",
            "form_type",
            "filing_date",
            "report_date",
            "accession_number",
        ]
    ].head()
)

Normalized corpus metadata successfully.


,ticker,company,cik,form_type,filing_date,report_date,accession_number
0,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021
1,NVDA,NVIDIA CORP,0001045810,10-K,2025-02-26,2025-01-26,0001045810-25-000023
2,NVDA,NVIDIA CORP,0001045810,10-K,2024-02-21,2024-01-28,0001045810-24-000029
3,NVDA,NVIDIA CORP,0001045810,10-Q,2026-05-20,2026-04-26,0001045810-26-000052
4,NVDA,NVIDIA CORP,0001045810,10-Q,2025-11-19,2025-10-26,0001045810-25-000230


Validate accession-number structure

In [ ]:
ACCESSION_PATTERN = re.compile(r"^\d{10}-\d{2}-\d{6}$")

invalid_accessions = frozen_corpus.loc[
    ~frozen_corpus["accession_number"].str.fullmatch(ACCESSION_PATTERN),
    ["ticker", "form_type", "accession_number"],
]

print(f"Invalid accession numbers: {len(invalid_accessions)}")

if not invalid_accessions.empty:
    display(invalid_accessions)

assert invalid_accessions.empty, "One or more accession numbers are malformed."

Invalid accession numbers: 0


Create deterministic IDs and content hashes

In [ ]:
def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def build_document_id(
    ticker: str,
    form_type: str,
    accession_number: str,
) -> str:
    normalized_form = form_type.lower().replace("-", "")
    normalized_accession = accession_number.replace("-", "")

    return f"{ticker.lower()}_{normalized_form}_{normalized_accession}"


frozen_corpus["document_id"] = frozen_corpus.apply(
    lambda row: build_document_id(
        ticker=row["ticker"],
        form_type=row["form_type"],
        accession_number=row["accession_number"],
    ),
    axis=1,
)

frozen_corpus["content_sha256"] = frozen_corpus["text"].map(sha256_text)
frozen_corpus["raw_char_count"] = frozen_corpus["text"].str.len()
frozen_corpus["raw_word_count"] = frozen_corpus["text"].str.split().str.len()

assert frozen_corpus["document_id"].is_unique
assert frozen_corpus["content_sha256"].notna().all()
assert frozen_corpus["content_sha256"].str.fullmatch(r"[0-9a-f]{64}").all()

print(f"Documents: {len(frozen_corpus):,}")
print(
    "Duplicate document IDs:",
    frozen_corpus["document_id"].duplicated().sum(),
)
print(
    "Duplicate content hashes:",
    frozen_corpus["content_sha256"].duplicated().sum(),
)

display(
    frozen_corpus[
        [
            "document_id",
            "accession_number",
            "content_sha256",
            "raw_char_count",
            "raw_word_count",
        ]
    ].head()
)

Documents: 30
Duplicate document IDs: 0
Duplicate content hashes: 0


,document_id,accession_number,content_sha256,raw_char_count,raw_word_count
0,nvda_10k_000104581026000021,0001045810-26-000021,cfbff785c98bdb8de6af877b359e69cb8e218ff0405d1c...,340269,51979
1,nvda_10k_000104581025000023,0001045810-25-000023,9eaa624b62b2f2fd208d7a0e7ad6063f2fbe0e9ef7461e...,348341,53347
2,nvda_10k_000104581024000029,0001045810-24-000029,6fd3f0778e246de9a52a58f223a831608b6bd13054f433...,339967,52189
3,nvda_10q_000104581026000052,0001045810-26-000052,df2c9d1923213848aedc5fc7d19fcebf6958a24c825029...,136174,21159
4,nvda_10q_000104581025000230,0001045810-25-000230,4d351283e65714f9ca9c6a6c46aed94f42932b978cb61b...,151162,23692


Create deterministic IDs and content hashes

In [ ]:
def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def build_document_id(
    ticker: str,
    form_type: str,
    accession_number: str,
) -> str:
    normalized_form = form_type.lower().replace("-", "")
    normalized_accession = accession_number.replace("-", "")

    return f"{ticker.lower()}_{normalized_form}_{normalized_accession}"


frozen_corpus["document_id"] = frozen_corpus.apply(
    lambda row: build_document_id(
        ticker=row["ticker"],
        form_type=row["form_type"],
        accession_number=row["accession_number"],
    ),
    axis=1,
)

frozen_corpus["content_sha256"] = frozen_corpus["text"].map(sha256_text)
frozen_corpus["raw_char_count"] = frozen_corpus["text"].str.len()
frozen_corpus["raw_word_count"] = frozen_corpus["text"].str.split().str.len()

assert frozen_corpus["document_id"].is_unique
assert frozen_corpus["content_sha256"].notna().all()
assert frozen_corpus["content_sha256"].str.fullmatch(r"[0-9a-f]{64}").all()

print(f"Documents: {len(frozen_corpus):,}")
print(
    "Duplicate document IDs:",
    frozen_corpus["document_id"].duplicated().sum(),
)
print(
    "Duplicate content hashes:",
    frozen_corpus["content_sha256"].duplicated().sum(),
)

display(
    frozen_corpus[
        [
            "document_id",
            "accession_number",
            "content_sha256",
            "raw_char_count",
            "raw_word_count",
        ]
    ].head()
)

Documents: 30
Duplicate document IDs: 0
Duplicate content hashes: 0


,document_id,accession_number,content_sha256,raw_char_count,raw_word_count
0,nvda_10k_000104581026000021,0001045810-26-000021,cfbff785c98bdb8de6af877b359e69cb8e218ff0405d1c...,340269,51979
1,nvda_10k_000104581025000023,0001045810-25-000023,9eaa624b62b2f2fd208d7a0e7ad6063f2fbe0e9ef7461e...,348341,53347
2,nvda_10k_000104581024000029,0001045810-24-000029,6fd3f0778e246de9a52a58f223a831608b6bd13054f433...,339967,52189
3,nvda_10q_000104581026000052,0001045810-26-000052,df2c9d1923213848aedc5fc7d19fcebf6958a24c825029...,136174,21159
4,nvda_10q_000104581025000230,0001045810-25-000230,4d351283e65714f9ca9c6a6c46aed94f42932b978cb61b...,151162,23692


In [ ]:
MANIFEST_COLUMNS = [
    "document_id",
    "ticker",
    "company",
    "cik",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "primary_document",
    "source_url",
    "content_sha256",
    "raw_char_count",
    "raw_word_count",
]

corpus_manifest = (
    frozen_corpus[MANIFEST_COLUMNS]
    .sort_values(
        ["ticker", "form_type", "filing_date", "accession_number"],
        kind="stable",
    )
    .reset_index(drop=True)
)

fingerprint_records = (
    corpus_manifest[
        ["document_id", "accession_number", "content_sha256"]
    ]
    .sort_values("document_id")
    .to_dict(orient="records")
)

canonical_json = json.dumps(
    fingerprint_records,
    sort_keys=True,
    separators=(",", ":"),
)

corpus_sha256 = hashlib.sha256(
    canonical_json.encode("utf-8")
).hexdigest()

corpus_version = f"sec30_{corpus_sha256[:12]}"

print(f"Corpus version: {corpus_version}")
print(f"Corpus SHA-256: {corpus_sha256}")
display(corpus_manifest.head())

Corpus version: sec30_959ac4a68dbf
Corpus SHA-256: 959ac4a68dbfaf74dbd667bf9da8520d4727516c26776e46d9f05ba17cda1ff9


,document_id,ticker,company,cik,form_type,filing_date,report_date,accession_number,primary_document,source_url,content_sha256,raw_char_count,raw_word_count
0,aapl_10k_000032019323000106,AAPL,Apple Inc.,0000320193,10-K,2023-11-03,2023-09-30,0000320193-23-000106,aapl-20230930.htm,https://www.sec.gov/Archives/edgar/data/320193...,06fffabc25f99c8e9b02d81530df9df87fa476244c51d8...,200598,30650
1,aapl_10k_000032019324000123,AAPL,Apple Inc.,0000320193,10-K,2024-11-01,2024-09-28,0000320193-24-000123,aapl-20240928.htm,https://www.sec.gov/Archives/edgar/data/320193...,48dadd62ed7cfdcdf2f0818762cf9eba9021b9db46afba...,204321,31122
2,aapl_10k_000032019325000079,AAPL,Apple Inc.,0000320193,10-K,2025-10-31,2025-09-27,0000320193-25-000079,aapl-20250927.htm,https://www.sec.gov/Archives/edgar/data/320193...,79983dc36aac906b94096f41d4c19f224bbc91ac3959e3...,206727,31482
3,aapl_10q_000032019325000073,AAPL,Apple Inc.,0000320193,10-Q,2025-08-01,2025-06-28,0000320193-25-000073,aapl-20250628.htm,https://www.sec.gov/Archives/edgar/data/320193...,49d345b52797b5b56f9e030978f8b1efe17ba22df9fffc...,59899,9689
4,aapl_10q_000032019326000006,AAPL,Apple Inc.,0000320193,10-Q,2026-01-30,2025-12-27,0000320193-26-000006,aapl-20251227.htm,https://www.sec.gov/Archives/edgar/data/320193...,dda53b2a795a0a78f44853062e95021079137f2b5671e9...,56972,9033


saving baseline

In [ ]:
MANIFEST_DIR = PROJECT_ROOT / "data/manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV_PATH = (
    MANIFEST_DIR / f"corpus_manifest_{corpus_version}.csv"
)

corpus_manifest.to_csv(
    MANIFEST_CSV_PATH,
    index=False,
)

print(f"Saved manifest: {MANIFEST_CSV_PATH}")
print(f"Rows: {len(corpus_manifest):,}")
print(f"Columns: {len(corpus_manifest.columns)}")
print(f"Size: {MANIFEST_CSV_PATH.stat().st_size:,} bytes")

Saved manifest: /content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_manifest_sec30_959ac4a68dbf.csv
Rows: 30
Columns: 13
Size: 8,742 bytes


In [ ]:
corpus_metadata = {
    "corpus_version": corpus_version,
    "corpus_sha256": corpus_sha256,
    "document_count": int(len(corpus_manifest)),
    "ticker_count": int(corpus_manifest["ticker"].nunique()),
    "tickers": sorted(corpus_manifest["ticker"].unique().tolist()),
    "forms": sorted(corpus_manifest["form_type"].unique().tolist()),
    "documents_by_ticker_and_form": {
        f"{ticker}_{form_type}": int(count)
        for (ticker, form_type), count in (
            corpus_manifest
            .groupby(["ticker", "form_type"])
            .size()
            .items()
        )
    },
    "manifest_filename": MANIFEST_CSV_PATH.name,
    "source_artifact": str(LEGACY_RAW_CSV.relative_to(PROJECT_ROOT)),
    "fingerprint_fields": [
        "document_id",
        "accession_number",
        "content_sha256",
    ],
}

METADATA_JSON_PATH = (
    MANIFEST_DIR / f"corpus_metadata_{corpus_version}.json"
)

with METADATA_JSON_PATH.open("w", encoding="utf-8") as file:
    json.dump(corpus_metadata, file, indent=2, sort_keys=True)

print(f"Saved metadata: {METADATA_JSON_PATH}")

Saved metadata: /content/drive/MyDrive/finance-rag-analyst/data/manifests/corpus_metadata_sec30_959ac4a68dbf.json


In [ ]:
saved_manifest = pd.read_csv(
    MANIFEST_CSV_PATH,
    dtype={"cik": "string"},
)

with METADATA_JSON_PATH.open("r", encoding="utf-8") as file:
    saved_metadata = json.load(file)

assert len(saved_manifest) == 30
assert saved_manifest["document_id"].is_unique
assert saved_manifest["accession_number"].is_unique
assert saved_metadata["corpus_version"] == corpus_version
assert saved_metadata["corpus_sha256"] == corpus_sha256

print("Saved artifacts verified successfully.")
print(f"Manifest: {MANIFEST_CSV_PATH.name}")
print(f"Metadata: {METADATA_JSON_PATH.name}")
print(f"Corpus version: {saved_metadata['corpus_version']}")

Saved artifacts verified successfully.
Manifest: corpus_manifest_sec30_959ac4a68dbf.csv
Metadata: corpus_metadata_sec30_959ac4a68dbf.json
Corpus version: sec30_959ac4a68dbf
